# 01. Làm sạch dữ liệu CPI

**Mục tiêu:** đi từ dữ liệu gốc (bảng Excel 2006-2025 + file bổ sung 2026) đến một bảng sạch, mỗi dòng một tháng, đủ 250 quan sát (01/2006 đến 10/2026).

**Các bước:** (1) xem dữ liệu gốc → (2) chuyển bảng rộng sang dạng dài và gộp → (3) kiểm tra chất lượng → (4) dựng chuỗi chỉ số → (5) lưu kết quả.

Thời điểm đóng băng dữ liệu: 19/11/2026. Không dùng tháng 11 và 12/2026.

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cpi_forecast.data.loader import load_wide_table, load_supplement, load_cpi_monthly
from cpi_forecast.data.cleaning import check_quality, clean_cpi

XLSX = "../data/raw/cpi_2006_2025.xlsx"
CSV = "../data/raw/cpi_2026_bo_sung.csv"


## 1. Dữ liệu gốc

Hai nguồn, đều ở thang **"tháng trước = 100"** (ví dụ 100.05 nghĩa là CPI tăng 0,05% so với tháng trước).

In [2]:
# Bảng Excel 2006-2025, đọc nguyên trạng (chưa xử lý gì)
raw = pd.read_excel(XLSX, header=None)
print("Kích thước:", raw.shape)
raw.iloc[:6, :6]

Kích thước: (15, 21)


,0,1,2,3,4,5
0,Chỉ số giá tiêu dùng các tháng trong năm chia ...,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2006.0,2007.0,2008.00,2009.00,2010.00
3,Tháng 1,101.2,101.1,102.38,100.32,101.36
4,Tháng 2,102.1,102.2,103.56,101.17,101.96
5,Tháng 3,99.5,99.8,102.99,99.83,100.75


**Nhận xét:** đây là bảng **dạng rộng**: dòng 1 là tiêu đề, dòng 2 trống, dòng 3 là các năm, từ dòng 4 là "Tháng 1" đến "Tháng 12". Dạng này đẹp để đọc nhưng không dùng được cho chuỗi thời gian, vì mỗi dòng chứa 20 tháng khác nhau.

In [3]:
# File bổ sung 2026 (tự thu thập từ bản tin hằng tháng)
supp_raw = pd.read_csv(CSV)
supp_raw

,nam,thang,cpi_so_voi_thang_truoc,nguon,ngay_tai
0,2026,1,100.05,https://www.nso.gov.vn/du-lieu-va-so-lieu-thon...,19/11/2026
1,2026,2,101.14,https://www.nso.gov.vn/tin-tuc-thong-ke/2026/0...,19/11/2026
2,2026,3,101.23,https://www.nso.gov.vn/du-lieu-va-so-lieu-thon...,19/11/2026
3,2026,4,100.84,https://www.nso.gov.vn/du-lieu-va-so-lieu-thon...,19/11/2026
4,2026,5,100.29,https://www.nso.gov.vn/tin-tuc-thong-ke/2026/0...,19/11/2026
5,2026,6,99.61,https://www.nso.gov.vn/tin-tuc-thong-ke/2026/0...,19/11/2026
6,2026,7,99.88,https://www.nso.gov.vn/tin-tuc-thong-ke/2026/0...,19/11/2026
7,2026,8,100.47,https://www.nso.gov.vn/tin-tuc-thong-ke/2026/0...,19/11/2026
8,2026,9,100.62,https://www.nso.gov.vn/du-lieu-va-so-lieu-thon...,19/11/2026
9,2026,10,100.20,https://www.nso.gov.vn/du-lieu-va-so-lieu-thon...,19/11/2026


## 2. Chuyển sang dạng dài và gộp hai nguồn

Hàm `load_wide_table` dùng `melt` để "gập" các cột năm thành một cột, kết quả là **mỗi dòng một tháng**.

In [4]:
long_2006_2025 = load_wide_table(XLSX)
print("2006-2025:", long_2006_2025.shape)
long_2006_2025.head()

2006-2025: (240, 2)


,date,mom
0,2006-01-01,101.2
1,2006-02-01,102.1
2,2006-03-01,99.5
3,2006-04-01,100.2
4,2006-05-01,100.6


In [5]:
long_2026 = load_supplement(CSV)
print("2026:", long_2026.shape)
long_2026

2026: (10, 2)


,date,mom
0,2026-01-01,100.05
1,2026-02-01,101.14
2,2026-03-01,101.23
3,2026-04-01,100.84
4,2026-05-01,100.29
5,2026-06-01,99.61
6,2026-07-01,99.88
7,2026-08-01,100.47
8,2026-09-01,100.62
9,2026-10-01,100.20


In [6]:
# Gộp, sắp theo thời gian, cắt cứng tại mốc đóng băng 10/2026
cpi = load_cpi_monthly(xlsx_path=XLSX, csv_path=CSV, freeze="2026-10")
print("Sau khi gộp:", cpi.shape)
display(cpi.head(3))
display(cpi.tail(3))

Sau khi gộp: (250, 2)


,date,mom
0,2006-01-01,101.2
1,2006-02-01,102.1
2,2006-03-01,99.5


,date,mom
247,2026-08-01,100.47
248,2026-09-01,100.62
249,2026-10-01,100.20


## 3. Kiểm tra chất lượng

`check_quality` trả về danh sách vấn đề; **danh sách rỗng nghĩa là dữ liệu sạch**.

In [7]:
issues = check_quality(cpi, freeze="2026-10")
print("Vấn đề phát hiện:", issues)

tom_tat = pd.Series({
    "Số dòng": len(cpi),
    "Tháng đầu": cpi["date"].min().strftime("%m/%Y"),
    "Tháng cuối": cpi["date"].max().strftime("%m/%Y"),
    "Số giá trị thiếu": int(cpi["mom"].isna().sum()),
    "Số tháng trùng": int(cpi["date"].duplicated().sum()),
    "mom nhỏ nhất": cpi["mom"].min(),
    "mom lớn nhất": cpi["mom"].max(),
})
tom_tat

Vấn đề phát hiện: []


Số dòng                 250
Tháng đầu           01/2006
Tháng cuối          10/2026
Số giá trị thiếu          0
Số tháng trùng            0
mom nhỏ nhất          98.46
mom lớn nhất         103.91
dtype: object

**Kiểm tra bộ kiểm tra:** để chắc `check_quality` thật sự bắt được lỗi, ta cố tình làm hỏng dữ liệu rồi xem nó có báo không.

In [8]:
# Làm mất một tháng
bi_thieu = cpi.drop(index=100)
print("Thiếu tháng ->", check_quality(bi_thieu))

# Gõ nhầm 100.05 thành 10.05
bi_sai = cpi.copy()
bi_sai.loc[5, "mom"] = 10.05
print("Giá trị lạ  ->", check_quality(bi_sai))

Thiếu tháng -> ["Thiếu tháng: ['2014-05']"]
Giá trị lạ  -> ["mom ngoài khoảng [97.0, 105.0]: ['2006-06']"]


**Ghi chú về độ chính xác:** thử đếm số chữ số thập phân lớn nhất của từng năm.

In [9]:
so_chu_so = cpi["mom"].astype(str).str.split(".").str[1].str.len()
cpi.assign(nam=cpi["date"].dt.year, so_chu_so=so_chu_so).groupby("nam")["so_chu_so"].max().head(5)

nam
2006    1
2007    1
2008    2
2009    2
2010    2
Name: so_chu_so, dtype: int64

Năm 2006 và 2007 chỉ có tối đa **1 chữ số thập phân**, từ 2008 trở đi có 2. Ta giữ nguyên số của nguồn, không "làm đẹp" lại.

## 4. Dựng chuỗi chỉ số

Dữ liệu gốc là tỷ lệ thay đổi theo tháng, chưa phải chỉ số liên tục. Ta nhân dồn với kỳ gốc **01/2006 = 100**:

`index(t) = index(t-1) × mom(t) / 100`

`clean_cpi` kiểm tra chất lượng rồi thêm hai cột: `pct = mom − 100` và `index`.

In [10]:
clean = clean_cpi(cpi, freeze="2026-10")
clean.head()

,date,mom,pct,index
0,2006-01-01,101.2,1.2,100.000000
1,2006-02-01,102.1,2.1,102.100000
2,2006-03-01,99.5,-0.5,101.589500
3,2006-04-01,100.2,0.2,101.792679
4,2006-05-01,100.6,0.6,102.403435


In [11]:
# Kiểm tra bằng tay: tháng 2/2006 có mom = 102.1 -> index = 100 * 1.021 = 102.1
print("index tháng 1/2006:", clean["index"].iloc[0])
print("index tháng 2/2006:", clean["index"].iloc[1], "(tính tay:", 100 * 102.1 / 100, ")")

# Kiểm tra nhất quán: chia index liên tiếp phải ra lại đúng mom
lech = (clean["index"] / clean["index"].shift(1) * 100 - clean["mom"]).abs().max()
print("Lệch lớn nhất so với mom:", lech)
print("index tháng 10/2026:", round(clean["index"].iloc[-1], 2))

index tháng 1/2006: 100.0
index tháng 2/2006: 102.1 (tính tay: 102.1 )
Lệch lớn nhất so với mom: 2.842170943040401e-14
index tháng 10/2026: 332.26


## 5. Lưu dữ liệu đã xử lý

In [12]:
OUT = "../data/processed/cpi_monthly_clean.csv"
clean.to_csv(OUT, index=False, float_format="%.6f")

kiem_tra = pd.read_csv(OUT, parse_dates=["date"])
print("Đã lưu:", kiem_tra.shape, "| cột:", list(kiem_tra.columns))
kiem_tra.tail(3)

Đã lưu: (250, 4) | cột: ['date', 'mom', 'pct', 'index']


,date,mom,pct,index
247,2026-08-01,100.47,0.47,329.551759
248,2026-09-01,100.62,0.62,331.594980
249,2026-10-01,100.20,0.20,332.258170


## Tóm tắt

- 250 quan sát, 01/2006 đến 10/2026, mỗi tháng đúng một dòng, không thiếu, không trùng.
- Bốn cột: `date`, `mom` (tháng trước = 100), `pct` (= mom − 100), `index` (kỳ gốc 01/2006 = 100).
- Năm 2006 và 2007 chỉ có 1 chữ số thập phân; giữ nguyên số của nguồn.
- File kết quả: `data/processed/cpi_monthly_clean.csv`, dùng cho các notebook 02 đến 05.